## ANN with bagging and stacking

In [4]:
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_predict
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, r2_score

# df = pd.read_excel(r'updated_fc_predictions.xlsx', sheet_name='Sheet1')
# df.dropna(inplace=True)

# X = df[['AGE', 'PC', 'PC_TYPE', 'FA', 'SS', 'SF', 'FAGG', 'CAGG', 'WATER', 'AEA', 'WR_HR', 'WR', 'ACC', 
#         'VOID', 'w/b', 'b/a', 'CAGG%', 'FAGG%', 'FA%', 'SS%', 'SF%']].values

# Y = df['fc (MPa)'].values

# Load and preprocess the data
df = pd.read_excel(r'PA Concrete Database with GWP cement type 5.17.2024.xlsx', sheet_name='Sheet1')
df.dropna(inplace=True)
# Define features and target
X = df[['AGE', 'PC', 'PC_TYPE', 'FA', 'SS', 'SF', 'FAGG', 'CAGG', 'WATER', 'AEA', 'WR_HR', 'WR', 'ACC', 
        'VOID', 'w/b', 'b/a', 'CAGG%', 'FAGG%', 'FA%', 'SS%', 'SF%']].values
y = df['GWP/strength'].values

# Split the data into training and testing sets (20% test data)
X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Bagging - Resample 80% of the training data
X_train, _, y_train, _ = train_test_split(X_train_full, y_train_full, train_size=0.8, random_state=42)

# Base models (Random Forest, XGBoost, CatBoost, LightGBMXT, ANN)
rf_reg = RandomForestRegressor(bootstrap=False, max_depth=30, max_features='sqrt',
                      min_samples_split=10, n_estimators=1800)
xgb_reg = XGBRegressor(base_score=0.5, booster='gbtree', callbacks=None,
             colsample_bylevel=1, colsample_bynode=1, colsample_bytree=0.7,
             early_stopping_rounds=None, enable_categorical=False,
             eval_metric=None, gamma=0.0, gpu_id=-1, grow_policy='depthwise',
             importance_type=None, interaction_constraints='',
             learning_rate=0.09, max_bin=256, max_cat_to_onehot=4,
             max_delta_step=0, max_depth=5, max_leaves=0, min_child_weight=1, monotone_constraints='()', n_estimators=500, n_jobs=0,
             num_parallel_tree=1, predictor='auto', random_state=42, reg_alpha=0,
             reg_lambda=1)
catboost_reg = CatBoostRegressor(silent=True,learning_rate=0.08, l2_leaf_reg= 6, iterations=500, depth=6, border_count=48, random_state=42)
lgbmxt_reg = LGBMRegressor(boosting_type='goss', reg_lambda=0.3, reg_alpha=0.4, num_leaves=40, n_estimators=500, min_child_samples= 45, 
learning_rate= 0.06, colsample_bytree=0.5,random_state=42)
ann_reg_base = MLPRegressor(solver= 'adam', max_iter=300, learning_rate_init=0.001, learning_rate='adaptive', 
hidden_layer_sizes=(50,), alpha=0.01, activation='relu', random_state=42)

# Perform Cross-validation with bagging for each base model
rf_pred = cross_val_predict(rf_reg, X_train, y_train, cv=5)
xgb_pred = cross_val_predict(xgb_reg, X_train, y_train, cv=5)
catboost_pred = cross_val_predict(catboost_reg, X_train, y_train, cv=5)
lgbmxt_pred = cross_val_predict(lgbmxt_reg, X_train, y_train, cv=5)
ann_pred = cross_val_predict(ann_reg_base, X_train, y_train, cv=5)

# Combine predictions as new features for the meta-model
stacked_features = pd.DataFrame({
    'rf_pred': rf_pred.ravel(),  # Ensuring 1-dimensional
    'xgb_pred': xgb_pred.ravel(),
    'catboost_pred': catboost_pred.ravel(),
    'lgbmxt_pred': lgbmxt_pred.ravel(),
    'ann_pred': ann_pred.ravel()
})

# Meta-model: ANN for regression
meta_reg = MLPRegressor(hidden_layer_sizes=(100,), max_iter=500, random_state=42)
meta_reg.fit(stacked_features, y_train)

# Predict on the test set using the stacked model
rf_test_pred = rf_reg.fit(X_train, y_train).predict(X_test)
xgb_test_pred = xgb_reg.fit(X_train, y_train).predict(X_test)
catboost_test_pred = catboost_reg.fit(X_train, y_train).predict(X_test)
lgbmxt_test_pred = lgbmxt_reg.fit(X_train, y_train).predict(X_test)
ann_test_pred = ann_reg_base.fit(X_train, y_train).predict(X_test)

# Combine test set predictions for final evaluation
stacked_test_features = pd.DataFrame({
    'rf_pred': rf_test_pred.ravel(),
    'xgb_pred': xgb_test_pred.ravel(),
    'catboost_pred': catboost_test_pred.ravel(),
    'lgbmxt_pred': lgbmxt_test_pred.ravel(),
    'ann_pred': ann_test_pred.ravel()
})

# Final meta-model prediction on the test set
y_pred_stacked = meta_reg.predict(stacked_test_features)

# Evaluate the performance of the stacked model
mse = mean_squared_error(y_test, y_pred_stacked)
r2 = r2_score(y_test, y_pred_stacked)

# Display the results
print(f"Stacking Regressor with Bagging and Cross-Validation Report:\nMSE: {mse:.4f}\nR²: {r2:.4f}")


/Users/zhangtianjie/opt/anaconda3/envs/ag/lib/python3.10/site-packages/xgboost/core.py:158: UserWarning: [13:05:53] WARNING: /Users/runner/miniforge3/conda-bld/xgboost-split_1727634900191/work/src/learner.cc:740: 
Parameters: { "predictor" } are not used.

  warnings.warn(smsg, UserWarning)
/Users/zhangtianjie/opt/anaconda3/envs/ag/lib/python3.10/site-packages/xgboost/core.py:158: UserWarning: [13:05:54] WARNING: /Users/runner/miniforge3/conda-bld/xgboost-split_1727634900191/work/src/learner.cc:740: 
Parameters: { "predictor" } are not used.

  warnings.warn(smsg, UserWarning)
/Users/zhangtianjie/opt/anaconda3/envs/ag/lib/python3.10/site-packages/xgboost/core.py:158: UserWarning: [13:05:55] WARNING: /Users/runner/miniforge3/conda-bld/xgboost-split_1727634900191/work/src/learner.cc:740: 
Parameters: { "predictor" } are not used.

  warnings.warn(smsg, UserWarning)
/Users/zhangtianjie/opt/anaconda3/envs/ag/lib/python3.10/site-packages/xgboost/core.py:158: UserWarning: [13:05:57] WARNING:

[LightGBM] [Warning] Found boosting=goss. For backwards compatibility reasons, LightGBM interprets this as boosting=gbdt, data_sample_strategy=goss.To suppress this warning, set data_sample_strategy=goss instead.
[LightGBM] [Warning] Found boosting=goss. For backwards compatibility reasons, LightGBM interprets this as boosting=gbdt, data_sample_strategy=goss.To suppress this warning, set data_sample_strategy=goss instead.
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000457 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2708
[LightGBM] [Info] Number of data points in the train set: 3473, number of used features: 21
[LightGBM] [Info] Using GOSS
[LightGBM] [Info] Start training from score 14.860489
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive 

/Users/zhangtianjie/opt/anaconda3/envs/ag/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/zhangtianjie/opt/anaconda3/envs/ag/lib/python3.10/site-packages/xgboost/core.py:158: UserWarning: [13:06:27] WARNING: /Users/runner/miniforge3/conda-bld/xgboost-split_1727634900191/work/src/learner.cc:740: 
Parameters: { "predictor" } are not used.

  warnings.warn(smsg, UserWarning)


[LightGBM] [Warning] Found boosting=goss. For backwards compatibility reasons, LightGBM interprets this as boosting=gbdt, data_sample_strategy=goss.To suppress this warning, set data_sample_strategy=goss instead.
[LightGBM] [Warning] Found boosting=goss. For backwards compatibility reasons, LightGBM interprets this as boosting=gbdt, data_sample_strategy=goss.To suppress this warning, set data_sample_strategy=goss instead.
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000556 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2829
[LightGBM] [Info] Number of data points in the train set: 4342, number of used features: 21
[LightGBM] [Info] Using GOSS
[LightGBM] [Info] Start training from score 14.870979
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive 

In [5]:
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_predict, KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from sklearn.metrics import mean_squared_error, r2_score

# Load dataset
df = pd.read_excel(r'updated_fc_predictions.xlsx', sheet_name='Sheet1')
df.dropna(inplace=True)

X = df[['AGE', 'PC', 'PC_TYPE', 'FA', 'SS', 'SF', 'FAGG', 'CAGG', 'WATER', 'AEA', 'WR_HR', 'WR', 'ACC', 
        'VOID', 'w/b', 'b/a', 'CAGG%', 'FAGG%', 'FA%', 'SS%', 'SF%']].values

y = df['fc (MPa)'].values

# Split the data into training and testing sets (20% test data)
X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Bagging - Resample 80% of the training data
X_train, _, y_train, _ = train_test_split(X_train_full, y_train_full, train_size=0.8, random_state=42)

# Base models (Random Forest, XGBoost, CatBoost, LightGBMXT)
rf_reg = RandomForestRegressor(n_estimators=200, random_state=42)
xgb_reg = XGBRegressor(random_state=42)
catboost_reg = CatBoostRegressor(silent=True, random_state=42)
lgbmxt_reg = LGBMRegressor(boosting_type='goss', random_state=42)

# Perform Cross-validation with bagging for each base model
rf_pred = cross_val_predict(rf_reg, X_train, y_train, cv=5)
xgb_pred = cross_val_predict(xgb_reg, X_train, y_train, cv=5)
catboost_pred = cross_val_predict(catboost_reg, X_train, y_train, cv=5)
lgbmxt_pred = cross_val_predict(lgbmxt_reg, X_train, y_train, cv=5)

# Combine original features and predictions as new features for the meta-model
stacked_features = np.concatenate([X_train, rf_pred.reshape(-1, 1), xgb_pred.reshape(-1, 1), 
                                   catboost_pred.reshape(-1, 1), lgbmxt_pred.reshape(-1, 1)], axis=1)

# Convert stacked features to PyTorch tensors
X_train_tensor = torch.tensor(stacked_features, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)

# Fit each of the base models on the training data
rf_reg.fit(X_train, y_train)
xgb_reg.fit(X_train, y_train)
catboost_reg.fit(X_train, y_train)
lgbmxt_reg.fit(X_train, y_train)

# Predict on the test set using the fitted base models
X_test_stacked = np.concatenate([X_test, rf_reg.predict(X_test).reshape(-1, 1), 
                                 xgb_reg.predict(X_test).reshape(-1, 1),
                                 catboost_reg.predict(X_test).reshape(-1, 1),
                                 lgbmxt_reg.predict(X_test).reshape(-1, 1)], axis=1)

X_test_tensor = torch.tensor(X_test_stacked, dtype=torch.float32)

# Custom ANN Model
class RegressionModel(nn.Module):
    def __init__(self, input_dim, layers, neurons):
        super(RegressionModel, self).__init__()
        self.layers = nn.ModuleList()
        self.layers.append(nn.Linear(input_dim, neurons))
        self.layers.append(nn.ReLU())
        for _ in range(layers - 1):
            self.layers.append(nn.Linear(neurons, neurons))
            self.layers.append(nn.ReLU())
        self.layers.append(nn.Linear(neurons, 1))

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

# Custom loss function using the fitted parameters
def custom_loss(outputs, targets, inputs, a, b, e, d):
    mse_loss = nn.MSELoss()(outputs, targets)
    
    AGE = inputs[:, 0]  # Assuming AGE is the first feature
    wb = inputs[:, -11]  # Assuming wb is the seventh feature from the end
    
    AGE = torch.clamp(AGE, min=1e-6)  # Clamp to avoid log of zero
    
    fc_pred = (a * torch.log(AGE) + b) * (e * torch.pow(AGE, d)) ** (-wb)
    residual = torch.abs(outputs - fc_pred.unsqueeze(1))
    residual = torch.nan_to_num(residual, nan=0.0, posinf=1e10, neginf=-1e10)
    
    mean_square_residual = torch.mean(residual ** 2)
    if mean_square_residual.item() > 0:
        residual_normalized = residual * torch.sqrt(mse_loss / mean_square_residual)
    else:
        residual_normalized = residual

    total_loss = 0.5 * mse_loss + 0.5 * torch.mean(residual_normalized)
    return total_loss

# Training function for the meta model
def train_model(model, optimizer, Xtrain, ytrain, epochs=300, batch_size=24, a=None, b=None, e=None, d=None):
    dataset = torch.utils.data.TensorDataset(Xtrain, ytrain)
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)
    
    for epoch in range(epochs):
        model.train()
        for inputs, targets in dataloader:
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = custom_loss(outputs, targets, inputs, a, b, e, d)
            loss.backward()
            optimizer.step()

# Build and train the custom ANN model (meta-model)
input_dim = stacked_features.shape[1]  # Number of features from the original features and base model predictions
model = RegressionModel(input_dim=input_dim, layers=3, neurons=232)
optimizer = optim.Adam(model.parameters(), lr=0.0076)

# Train the model
train_model(model, optimizer, X_train_tensor, y_train_tensor, epochs=300, batch_size=24, a=40.50, b=15.29, e=6.49, d=0.36)

# Use the trained model for predictions on the test set
model.eval()
with torch.no_grad():
    y_pred_stacked = model(X_test_tensor).numpy().flatten()

# Evaluate the performance
mse = mean_squared_error(y_test, y_pred_stacked)
r2 = r2_score(y_test, y_pred_stacked)

# Display the results
print(f"Stacking Regressor with Custom ANN Model (with original features) Report:\nMSE: {mse:.4f}\nR²: {r2:.4f}")


[LightGBM] [Warning] Found boosting=goss. For backwards compatibility reasons, LightGBM interprets this as boosting=gbdt, data_sample_strategy=goss.To suppress this warning, set data_sample_strategy=goss instead.
[LightGBM] [Warning] Found boosting=goss. For backwards compatibility reasons, LightGBM interprets this as boosting=gbdt, data_sample_strategy=goss.To suppress this warning, set data_sample_strategy=goss instead.
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000472 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2709
[LightGBM] [Info] Number of data points in the train set: 3456, number of used features: 21
[LightGBM] [Info] Using GOSS
[LightGBM] [Info] Start training from score 41.596530
[LightGBM] [Warning] Found boosting=goss. For backwards compatibility reasons, LightGBM interprets this as boosting=gbdt, data_sample_strategy=goss.To suppress this warning, set data_sample_strategy=goss

In [9]:
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from sklearn.metrics import mean_squared_error, r2_score

# Load dataset
df = pd.read_excel(r'updated_fc_predictions.xlsx', sheet_name='Sheet1')
df.dropna(inplace=True)

X = df[['AGE', 'PC', 'PC_TYPE', 'FA', 'SS', 'SF', 'FAGG', 'CAGG', 'WATER', 'AEA', 'WR_HR', 'WR', 'ACC', 
        'VOID', 'w/b', 'b/a', 'CAGG%', 'FAGG%', 'FA%', 'SS%', 'SF%']].values

y = df['fc (MPa)'].values

# Split the data into training and testing sets (20% test data)
X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Bagging - Resample 80% of the training data
X_train, _, y_train, _ = train_test_split(X_train_full, y_train_full, train_size=0.8, random_state=42)

# Base models (Random Forest, XGBoost, CatBoost, LightGBMXT)
rf_reg = RandomForestRegressor(n_estimators=200, random_state=42)
xgb_reg = XGBRegressor(random_state=42)
catboost_reg = CatBoostRegressor(silent=True, random_state=42)
lgbmxt_reg = LGBMRegressor(boosting_type='goss', random_state=42)

# Fit the base models
rf_reg.fit(X_train, y_train)
xgb_reg.fit(X_train, y_train)
catboost_reg.fit(X_train, y_train)
lgbmxt_reg.fit(X_train, y_train)

# Predictions for training data (cross-validation not done here for simplicity)
rf_pred = rf_reg.predict(X_train)
xgb_pred = xgb_reg.predict(X_train)
catboost_pred = catboost_reg.predict(X_train)
lgbmxt_pred = lgbmxt_reg.predict(X_train)

# Combine original features and predictions as new features for the meta-model
stacked_features = np.concatenate([X_train, rf_pred.reshape(-1, 1), xgb_pred.reshape(-1, 1), 
                                   catboost_pred.reshape(-1, 1), lgbmxt_pred.reshape(-1, 1)], axis=1)
print(stacked_features.shape)
# Convert stacked features to PyTorch tensors
X_train_tensor = torch.tensor(stacked_features, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)

# Prepare K-Fold Cross-Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Custom ANN Model
class RegressionModel(nn.Module):
    def __init__(self, input_dim, layers, neurons):
        super(RegressionModel, self).__init__()
        self.layers = nn.ModuleList()
        self.layers.append(nn.Linear(input_dim, neurons))
        self.layers.append(nn.ReLU())
        for _ in range(layers - 1):
            self.layers.append(nn.Linear(neurons, neurons))
            self.layers.append(nn.ReLU())
        self.layers.append(nn.Linear(neurons, 1))

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

# Custom loss function using the fitted parameters
def custom_loss(outputs, targets, inputs, a, b, e, d):
    mse_loss = nn.MSELoss()(outputs, targets)
    
    AGE = inputs[:, 0]  # Assuming AGE is the first feature
    wb = inputs[:, 14]  # Assuming wb is the seventh feature from the end
    
    AGE = torch.clamp(AGE, min=1e-6)  # Clamp to avoid log of zero
    
    fc_pred = (a * torch.log(AGE) + b) * (e * torch.pow(AGE, d)) ** (-wb)
    residual = torch.abs(outputs - fc_pred.unsqueeze(1))
    residual = torch.nan_to_num(residual, nan=0.0, posinf=1e3, neginf=-1e6)
    
    mean_square_residual = torch.mean(residual ** 2)
    if mean_square_residual.item() > 0:
        residual_normalized = residual * torch.sqrt(mse_loss / mean_square_residual)
    else:
        residual_normalized = residual

    total_loss = 0.5 * mse_loss + 0.5 * torch.mean(residual_normalized)
    return total_loss

# Training function for the meta model
def train_model(model, optimizer, Xtrain, ytrain, epochs=300, batch_size=24, a=None, b=None, e=None, d=None):
    dataset = torch.utils.data.TensorDataset(Xtrain, ytrain)
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)
    
    for epoch in range(epochs):
        model.train()
        for inputs, targets in dataloader:
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = custom_loss(outputs, targets, inputs, a, b, e, d)
            loss.backward()
            optimizer.step()

# AdaBoosting weights
boost_weights = np.ones(len(y_train_tensor)) / len(y_train_tensor)

# Store multiple models from cross-validation
meta_models = []
test_preds = []

# Build and train meta-model with K-Fold Cross-Validation and AdaBoosting
input_dim = stacked_features.shape[1]  # Number of features
for fold, (train_index, val_index) in enumerate(kf.split(X_train_tensor)):
    print(f"Fold {fold + 1}")
    
    X_train_fold = X_train_tensor[train_index]
    y_train_fold = y_train_tensor[train_index]
    
    # Build and initialize the meta model
    model = RegressionModel(input_dim=input_dim, layers=3, neurons=52)
    optimizer = optim.Adam(model.parameters(), lr=0.0076)
    
    # Train the model on this fold
    train_model(model, optimizer, X_train_fold, y_train_fold, epochs=300, batch_size=24, a=40.50, b=15.29, e=6.49, d=0.36)
    
    # Store the model
    meta_models.append(model)

# Test-time evaluation on the test set using each of the cross-validated models
X_test_stacked = np.concatenate([X_test, rf_reg.predict(X_test).reshape(-1, 1), 
                                 xgb_reg.predict(X_test).reshape(-1, 1),
                                 catboost_reg.predict(X_test).reshape(-1, 1),
                                 lgbmxt_reg.predict(X_test).reshape(-1, 1)], axis=1)
X_test_tensor = torch.tensor(X_test_stacked, dtype=torch.float32)

# Predict on the test set using all cross-validated models and average the results
meta_preds = np.zeros((len(X_test_tensor), len(meta_models)))
with torch.no_grad():
    for i, model in enumerate(meta_models):
        meta_preds[:, i] = model(X_test_tensor).numpy().flatten()

# Average the predictions across all models
y_pred_stacked = np.mean(meta_preds, axis=1)

# Evaluate the performance
mse = mean_squared_error(y_test, y_pred_stacked)
r2 = r2_score(y_test, y_pred_stacked)

# Display the results
print(f"Stacking Regressor with Custom ANN Model (AdaBoost + Cross-Validation) Report:\nMSE: {mse:.4f}\nR²: {r2:.4f}")


[LightGBM] [Warning] Found boosting=goss. For backwards compatibility reasons, LightGBM interprets this as boosting=gbdt, data_sample_strategy=goss.To suppress this warning, set data_sample_strategy=goss instead.
[LightGBM] [Warning] Found boosting=goss. For backwards compatibility reasons, LightGBM interprets this as boosting=gbdt, data_sample_strategy=goss.To suppress this warning, set data_sample_strategy=goss instead.
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000702 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2803
[LightGBM] [Info] Number of data points in the train set: 4321, number of used features: 21
[LightGBM] [Info] Using GOSS
[LightGBM] [Info] Start training from score 41.496497
[LightGBM] [Warning] Found boosting=goss. For backwards compatibility reasons, LightGBM interprets this as boosting=gbdt, data_sample_strategy=goss.To suppress this warning, set data_sample_strategy=goss

In [14]:
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_predict
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, r2_score

# Load dataset
df = pd.read_excel(r'updated_fc_predictions.xlsx', sheet_name='Sheet1')
df.dropna(inplace=True)

X = df[['AGE', 'PC', 'PC_TYPE', 'FA', 'SS', 'SF', 'FAGG', 'CAGG', 'WATER', 'AEA', 'WR_HR', 'WR', 'ACC', 
        'VOID', 'w/b', 'b/a', 'CAGG%', 'FAGG%', 'FA%', 'SS%', 'SF%']].values

y = df['fc (MPa)'].values

# Split the data into training and testing sets (20% test data)
X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Bagging - Resample 80% of the training data
X_train, _, y_train, _ = train_test_split(X_train_full, y_train_full, train_size=0.8, random_state=42)

# Base models (Random Forest, XGBoost, CatBoost, LightGBMXT, ANN)
rf_reg = RandomForestRegressor(n_estimators=200, random_state=42)
xgb_reg = XGBRegressor(random_state=42)
catboost_reg = CatBoostRegressor(silent=True, random_state=42)
lgbmxt_reg = LGBMRegressor(boosting_type='goss', random_state=42)
ann_reg_base = MLPRegressor(hidden_layer_sizes=(100,), max_iter=500, random_state=42)

# Perform Cross-validation with bagging for each base model
rf_pred = cross_val_predict(rf_reg, X_train, y_train, cv=5)
xgb_pred = cross_val_predict(xgb_reg, X_train, y_train, cv=5)
catboost_pred = cross_val_predict(catboost_reg, X_train, y_train, cv=5)
lgbmxt_pred = cross_val_predict(lgbmxt_reg, X_train, y_train, cv=5)
ann_pred = cross_val_predict(ann_reg_base, X_train, y_train, cv=5)

# Combine original features and base model predictions as new features for the meta-model
# stacked_features = pd.DataFrame(X_train, columns=['AGE', 'PC', 'PC_TYPE', 'FA', 'SS', 'SF', 'FAGG', 'CAGG', 'WATER', 'AEA', 'WR_HR', 'WR', 'ACC', 
#         'VOID', 'w/b', 'b/a', 'CAGG%', 'FAGG%', 'FA%', 'SS%', 'SF%'])  # Original features from X_train
stacked_features['rf_pred'] = rf_pred.ravel()
stacked_features['xgb_pred'] = xgb_pred.ravel()
stacked_features['catboost_pred'] = catboost_pred.ravel()
stacked_features['lgbmxt_pred'] = lgbmxt_pred.ravel()
stacked_features['ann_pred'] = ann_pred.ravel()

# Meta-model: ANN for regression
meta_reg = MLPRegressor(hidden_layer_sizes=(100,), max_iter=500, random_state=42)
meta_reg.fit(stacked_features, y_train)

# Predict on the test set using the stacked model
rf_test_pred = rf_reg.fit(X_train, y_train).predict(X_test)
xgb_test_pred = xgb_reg.fit(X_train, y_train).predict(X_test)
catboost_test_pred = catboost_reg.fit(X_train, y_train).predict(X_test)
lgbmxt_test_pred = lgbmxt_reg.fit(X_train, y_train).predict(X_test)
ann_test_pred = ann_reg_base.fit(X_train, y_train).predict(X_test)

# Combine original test set features and test set predictions for final evaluation
# stacked_test_features = pd.DataFrame(X_test, columns=['AGE', 'PC', 'PC_TYPE', 'FA', 'SS', 'SF', 'FAGG', 'CAGG', 'WATER', 'AEA', 'WR_HR', 'WR', 'ACC', 
#         'VOID', 'w/b', 'b/a', 'CAGG%', 'FAGG%', 'FA%', 'SS%', 'SF%'])  # Original features from X_test
stacked_test_features['rf_pred'] = rf_test_pred.ravel()
stacked_test_features['xgb_pred'] = xgb_test_pred.ravel()
stacked_test_features['catboost_pred'] = catboost_test_pred.ravel()
stacked_test_features['lgbmxt_pred'] = lgbmxt_test_pred.ravel()
stacked_test_features['ann_pred'] = ann_test_pred.ravel()

# Final meta-model prediction on the test set
y_pred_stacked = meta_reg.predict(stacked_test_features)

# Evaluate the performance of the stacked model
mse = mean_squared_error(y_test, y_pred_stacked)
r2 = r2_score(y_test, y_pred_stacked)

# Display the results
print(f"Stacking Regressor with Bagging and Cross-Validation Report (Original Features + Model Predictions):\nMSE: {mse:.4f}\nR²: {r2:.4f}")


[LightGBM] [Warning] Found boosting=goss. For backwards compatibility reasons, LightGBM interprets this as boosting=gbdt, data_sample_strategy=goss.To suppress this warning, set data_sample_strategy=goss instead.
[LightGBM] [Warning] Found boosting=goss. For backwards compatibility reasons, LightGBM interprets this as boosting=gbdt, data_sample_strategy=goss.To suppress this warning, set data_sample_strategy=goss instead.
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000484 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2709
[LightGBM] [Info] Number of data points in the train set: 3456, number of used features: 21
[LightGBM] [Info] Using GOSS
[LightGBM] [Info] Start training from score 41.596530
[LightGBM] [Warning] Found boosting=goss. For backwards compatibility reasons, LightGBM interprets this as boosting=gbdt, data_sample_strategy=goss.To suppress this warning, set data_sample_strategy=goss

In [12]:
stacked_features.head(10)

,AGE,PC,PC_TYPE,FA,SS,SF,FAGG,CAGG,WATER,AEA,...,CAGG%,FAGG%,FA%,SS%,SF%,rf_pred,xgb_pred,catboost_pred,lgbmxt_pred,ann_pred
0,28.0,420.0,8.0,0.0,280.0,40.0,1080.0,1760.0,271.0,7.4,...,0.619718,0.380282,0.000000,0.400000,0.057143,44.028723,49.172657,49.229205,50.966734,55.190867
1,56.0,340.0,6.0,0.0,225.0,0.0,1325.0,1800.0,255.0,0.0,...,0.576000,0.424000,0.000000,0.398230,0.000000,38.476696,43.441704,40.884729,40.178208,47.347868
2,28.0,390.0,11.0,0.0,260.0,0.0,1420.0,1600.0,242.0,14.0,...,0.529801,0.470199,0.000000,0.400000,0.000000,54.621195,57.125408,53.931233,52.021261,48.181224
3,28.0,485.0,1.0,0.0,160.0,0.0,1185.0,1800.0,266.0,11.0,...,0.603015,0.396985,0.000000,0.248062,0.000000,34.778903,35.452225,36.690211,37.649516,33.927667
4,28.0,560.0,2.0,100.0,0.0,0.0,1185.0,1850.0,264.0,10.0,...,0.609555,0.390445,0.151515,0.000000,0.000000,38.274648,38.002945,36.843868,37.719689,34.911824
5,7.0,564.0,11.0,188.0,0.0,0.0,1126.0,1695.0,267.0,3.6,...,0.600851,0.399149,0.250000,0.000000,0.000000,38.785159,44.040604,44.926030,42.051248,46.872748
6,28.0,400.0,2.0,0.0,259.0,0.0,1240.0,1755.0,241.0,8.0,...,0.585977,0.414023,0.000000,0.393020,0.000000,36.595258,37.431610,37.308648,39.810367,41.516703
7,28.0,198.0,11.0,198.0,264.0,0.0,1345.0,1620.0,249.0,5.0,...,0.546374,0.453626,0.300000,0.400000,0.000000,47.466238,50.443413,46.059650,45.661923,55.752608
8,56.0,600.0,2.0,224.0,0.0,0.0,1252.0,1642.0,317.0,6.7,...,0.567381,0.432619,0.271845,0.000000,0.000000,48.410780,55.277622,51.379122,50.201407,60.645686
9,7.0,476.0,11.0,204.0,0.0,0.0,1180.0,1750.0,254.0,28.0,...,0.597270,0.402730,0.300000,0.000000,0.000000,27.071911,28.037775,26.380000,26.368917,33.535497


In [13]:
y_train

array([47.68064518, 35.74243584, 56.85878747, ..., 39.75079859,
       30.56906759, 44.83466651])

In [19]:
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_predict
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, r2_score

# Load dataset
df = pd.read_excel(r'updated_fc_predictions.xlsx', sheet_name='Sheet1')
df.dropna(inplace=True)  # Drop rows with any NaN values

X = df[['AGE', 'PC', 'PC_TYPE', 'FA', 'SS', 'SF', 'FAGG', 'CAGG', 'WATER', 'AEA', 'WR_HR', 'WR', 'ACC', 
        'VOID', 'w/b', 'b/a', 'CAGG%', 'FAGG%', 'FA%', 'SS%', 'SF%']].values

y = df['fc (MPa)'].values

# Split the data into training and testing sets (20% test data)
X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Bagging - Resample 80% of the training data
X_train, _, y_train, _ = train_test_split(X_train_full, y_train_full, train_size=0.8, random_state=42)

# Base models (Random Forest, XGBoost, CatBoost, LightGBM, ANN)
rf_reg = RandomForestRegressor(n_estimators=200, random_state=42)
xgb_reg = XGBRegressor(random_state=42)
catboost_reg = CatBoostRegressor(silent=True, random_state=42)
lgbmxt_reg = LGBMRegressor(boosting_type='goss', random_state=42)
lightgbm_reg = LGBMRegressor(random_state=42)  # LightGBM base model
ann_reg_base = MLPRegressor(hidden_layer_sizes=(100,), max_iter=500, random_state=42)

# Perform Cross-validation with bagging for each base model
rf_pred = cross_val_predict(rf_reg, X_train, y_train, cv=5)
xgb_pred = cross_val_predict(xgb_reg, X_train, y_train, cv=5)
catboost_pred = cross_val_predict(catboost_reg, X_train, y_train, cv=5)
lgbmxt_pred = cross_val_predict(lgbmxt_reg, X_train, y_train, cv=5)
lightgbm_pred = cross_val_predict(lightgbm_reg, X_train, y_train, cv=5)  # LightGBM cross-validation
ann_pred = cross_val_predict(ann_reg_base, X_train, y_train, cv=5)

# Concatenate original features and model predictions correctly (horizontally)
stacked_features = pd.concat([pd.DataFrame(X_train, columns=['AGE', 'PC', 'PC_TYPE', 'FA', 'SS', 'SF', 'FAGG', 'CAGG', 'WATER', 'AEA', 'WR_HR', 'WR', 'ACC', 
        'VOID', 'w/b', 'b/a', 'CAGG%', 'FAGG%', 'FA%', 'SS%', 'SF%']),
    pd.DataFrame({
        'rf_pred': rf_pred.ravel(),
        'xgb_pred': xgb_pred.ravel(),
        'catboost_pred': catboost_pred.ravel(),
        'lgbmxt_pred': lgbmxt_pred.ravel(),
        'lightgbm_pred': lightgbm_pred.ravel(),  # Add LightGBM predictions
        'ann_pred': ann_pred.ravel()
    })
], axis=1)  # Concatenate horizontally

# Meta-model: ANN for regression
meta_reg = MLPRegressor(hidden_layer_sizes=(100,), max_iter=500, random_state=42)
meta_reg.fit(stacked_features, y_train)


# Predict on the test set using the stacked model
rf_test_pred = rf_reg.fit(X_train, y_train).predict(X_test)
xgb_test_pred = xgb_reg.fit(X_train, y_train).predict(X_test)
catboost_test_pred = catboost_reg.fit(X_train, y_train).predict(X_test)
lgbmxt_test_pred = lgbmxt_reg.fit(X_train, y_train).predict(X_test)
lightgbm_test_pred = lightgbm_reg.fit(X_train, y_train).predict(X_test)  # LightGBM test predictions
ann_test_pred = ann_reg_base.fit(X_train, y_train).predict(X_test)

# Concatenate original test set features and test set predictions for final evaluation
stacked_test_features = pd.concat([pd.DataFrame(X_test, columns=['AGE', 'PC', 'PC_TYPE', 'FA', 'SS', 'SF', 'FAGG', 'CAGG', 'WATER', 'AEA', 'WR_HR', 'WR', 'ACC', 
        'VOID', 'w/b', 'b/a', 'CAGG%', 'FAGG%', 'FA%', 'SS%', 'SF%']),
    pd.DataFrame({
        'rf_pred': rf_test_pred.ravel(),
        'xgb_pred': xgb_test_pred.ravel(),
        'catboost_pred': catboost_test_pred.ravel(),
        'lgbmxt_pred': lgbmxt_test_pred.ravel(),
        'lightgbm_pred': lightgbm_test_pred.ravel(),  # Add LightGBM test predictions
        'ann_pred': ann_test_pred.ravel()
    })
], axis=1)  # Concatenate horizontally

# Final meta-model prediction on the test set
y_pred_stacked = meta_reg.predict(stacked_test_features)

# Evaluate the performance of the stacked model
mse = mean_squared_error(y_test, y_pred_stacked)
r2 = r2_score(y_test, y_pred_stacked)

# Display the results
print(f"Stacking Regressor with Bagging and Cross-Validation Report (LightGBM Added):\nMSE: {mse:.4f}\nR²: {r2:.4f}")


[LightGBM] [Warning] Found boosting=goss. For backwards compatibility reasons, LightGBM interprets this as boosting=gbdt, data_sample_strategy=goss.To suppress this warning, set data_sample_strategy=goss instead.
[LightGBM] [Warning] Found boosting=goss. For backwards compatibility reasons, LightGBM interprets this as boosting=gbdt, data_sample_strategy=goss.To suppress this warning, set data_sample_strategy=goss instead.
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000523 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2709
[LightGBM] [Info] Number of data points in the train set: 3456, number of used features: 21
[LightGBM] [Info] Using GOSS
[LightGBM] [Info] Start training from score 41.596530
[LightGBM] [Warning] Found boosting=goss. For backwards compatibility reasons, LightGBM interprets this as boosting=gbdt, data_sample_strategy=goss.To suppress this warning, set data_sample_strategy=goss

In [ ]:
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_predict
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, r2_score

# Load dataset
df = pd.read_excel(r'updated_fc_predictions.xlsx', sheet_name='Sheet1')
df.dropna(inplace=True)  # Drop rows with any NaN values

X = df[['AGE', 'PC', 'PC_TYPE', 'FA', 'SS', 'SF', 'FAGG', 'CAGG', 'WATER', 'AEA', 'WR_HR', 'WR', 'ACC', 
        'VOID', 'w/b', 'b/a', 'CAGG%', 'FAGG%', 'FA%', 'SS%', 'SF%']].values

y = df['fc (MPa)'].values

# Split the data into training and testing sets (20% test data)
X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Bagging - Resample 80% of the training data
X_train, _, y_train, _ = train_test_split(X_train_full, y_train_full, train_size=0.8, random_state=42)

# Base models (Random Forest, XGBoost, CatBoost, LightGBM, ANN)
rf_reg = RandomForestRegressor(n_estimators=200, random_state=42)
xgb_reg = XGBRegressor(random_state=42)
catboost_reg = CatBoostRegressor(silent=True, random_state=42)
lgbmxt_reg = LGBMRegressor(boosting_type='goss', random_state=42)
lightgbm_reg = LGBMRegressor(random_state=42)  # LightGBM base model
ann_reg_base = MLPRegressor(hidden_layer_sizes=(100,), max_iter=500, random_state=42)

# Perform Cross-validation with bagging for each base model
rf_pred = cross_val_predict(rf_reg, X_train, y_train, cv=5)
xgb_pred = cross_val_predict(xgb_reg, X_train, y_train, cv=5)
catboost_pred = cross_val_predict(catboost_reg, X_train, y_train, cv=5)
lgbmxt_pred = cross_val_predict(lgbmxt_reg, X_train, y_train, cv=5)
lightgbm_pred = cross_val_predict(lightgbm_reg, X_train, y_train, cv=5)  # LightGBM cross-validation
ann_pred = cross_val_predict(ann_reg_base, X_train, y_train, cv=5)

# Concatenate original features and model predictions correctly (horizontally)
stacked_features = pd.concat([pd.DataFrame(X_train, columns=['AGE', 'PC', 'PC_TYPE', 'FA', 'SS', 'SF', 'FAGG', 'CAGG', 'WATER', 'AEA', 'WR_HR', 'WR', 'ACC', 
        'VOID', 'w/b', 'b/a', 'CAGG%', 'FAGG%', 'FA%', 'SS%', 'SF%']),
    pd.DataFrame({
        'rf_pred': rf_pred.ravel(),
        'xgb_pred': xgb_pred.ravel(),
        'catboost_pred': catboost_pred.ravel(),
        'lgbmxt_pred': lgbmxt_pred.ravel(),
        'lightgbm_pred': lightgbm_pred.ravel(),  # Add LightGBM predictions
        'ann_pred': ann_pred.ravel()
    })
], axis=1)  # Concatenate horizontally

# Meta-model: ANN for regression
meta_reg = MLPRegressor(hidden_layer_sizes=(100,), max_iter=500, random_state=42)
meta_reg.fit(stacked_features, y_train)


# Predict on the test set using the stacked model
rf_test_pred = rf_reg.fit(X_train, y_train).predict(X_test)
xgb_test_pred = xgb_reg.fit(X_train, y_train).predict(X_test)
catboost_test_pred = catboost_reg.fit(X_train, y_train).predict(X_test)
lgbmxt_test_pred = lgbmxt_reg.fit(X_train, y_train).predict(X_test)
lightgbm_test_pred = lightgbm_reg.fit(X_train, y_train).predict(X_test)  # LightGBM test predictions
ann_test_pred = ann_reg_base.fit(X_train, y_train).predict(X_test)

# Concatenate original test set features and test set predictions for final evaluation
stacked_test_features = pd.concat([pd.DataFrame(X_test, columns=['AGE', 'PC', 'PC_TYPE', 'FA', 'SS', 'SF', 'FAGG', 'CAGG', 'WATER', 'AEA', 'WR_HR', 'WR', 'ACC', 
        'VOID', 'w/b', 'b/a', 'CAGG%', 'FAGG%', 'FA%', 'SS%', 'SF%']),
    pd.DataFrame({
        'rf_pred': rf_test_pred.ravel(),
        'xgb_pred': xgb_test_pred.ravel(),
        'catboost_pred': catboost_test_pred.ravel(),
        'lgbmxt_pred': lgbmxt_test_pred.ravel(),
        'lightgbm_pred': lightgbm_test_pred.ravel(),  # Add LightGBM test predictions
        'ann_pred': ann_test_pred.ravel()
    })
], axis=1)  # Concatenate horizontally

# Final meta-model prediction on the test set
y_pred_stacked = meta_reg.predict(stacked_test_features)

# Evaluate the performance of the stacked model
mse = mean_squared_error(y_test, y_pred_stacked)
r2 = r2_score(y_test, y_pred_stacked)

# Display the results
print(f"Stacking Regressor with Bagging and Cross-Validation Report (LightGBM Added):\nMSE: {mse:.4f}\nR²: {r2:.4f}")


In [35]:
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_predict
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error, r2_score
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

# Load dataset
df = pd.read_excel(r'updated_fc_predictions.xlsx', sheet_name='Sheet1')
df.dropna(inplace=True)  # Drop rows with any NaN values

X = df[['AGE', 'PC', 'PC_TYPE', 'FA', 'SS', 'SF', 'FAGG', 'CAGG', 'WATER', 'AEA', 'WR_HR', 'WR', 'ACC', 
        'VOID', 'w/b', 'b/a', 'CAGG%', 'FAGG%', 'FA%', 'SS%', 'SF%']].values

y = df['fc (MPa)'].values

# Split the data into training and testing sets (20% test data)
X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Bagging - Resample 80% of the training data
X_train, _, y_train, _ = train_test_split(X_train_full, y_train_full, train_size=0.8, random_state=42)


from sklearn.model_selection import KFold
import numpy as np

# Define cross-validation strategy
kf = KFold(n_splits=10, shuffle=True, random_state=42)

# Initialize arrays to store averaged predictions on the training and test data
rf_train_predictions = np.zeros(len(X_train_full))
xgb_train_predictions = np.zeros(len(X_train_full))
catboost_train_predictions = np.zeros(len(X_train_full))
lgbmxt_train_predictions = np.zeros(len(X_train_full))
lightgbm_train_predictions = np.zeros(len(X_train_full))

rf_test_predictions = np.zeros(len(X_test))
xgb_test_predictions = np.zeros(len(X_test))
catboost_test_predictions = np.zeros(len(X_test))
lgbmxt_test_predictions = np.zeros(len(X_test))
lightgbm_test_predictions = np.zeros(len(X_test))

# Initialize base models
rf_reg = RandomForestRegressor(bootstrap=False, max_depth=30, max_features='sqrt',
                      min_samples_split=10, n_estimators=1800)
xgb_reg = XGBRegressor(base_score=0.5, booster='gbtree', callbacks=None,
             colsample_bylevel=1, colsample_bynode=1, colsample_bytree=0.7,
             early_stopping_rounds=None, enable_categorical=False,
             eval_metric=None, gamma=0.0, gpu_id=-1, grow_policy='depthwise',
             importance_type=None, interaction_constraints='',
             learning_rate=0.09, max_bin=256, max_cat_to_onehot=4,
             max_delta_step=0, max_depth=5, max_leaves=0, min_child_weight=1,
             missing=nan, monotone_constraints='()', n_estimators=500, n_jobs=0,
             num_parallel_tree=1, predictor='auto', random_state=0, reg_alpha=0,
             reg_lambda=1)
catboost_reg = CatBoostRegressor(silent=True, random_state=42)
lgbmxt_reg = LGBMRegressor(boosting_type='goss', random_state=42)
lightgbm_reg = LGBMRegressor(random_state=42)

# Manual cross-validation to get predictions for both train and test sets
for train_idx, val_idx in kf.split(X_train_full):
    # Split into training and validation based on the fold
    X_train_fold, X_val_fold = X_train_full[train_idx], X_train_full[val_idx]
    y_train_fold, y_val_fold = y_train_full[train_idx], y_train_full[val_idx]
    
    # Train Random Forest and make predictions on whole training data and X_test
    rf_reg.fit(X_train_fold, y_train_fold)
    rf_train_predictions += rf_reg.predict(X_train_full) / kf.n_splits  # Averaged predictions for whole training set
    rf_test_predictions += rf_reg.predict(X_test) / kf.n_splits  # Averaged predictions for test set
    
    # Train XGBoost and make predictions on whole training data and X_test
    xgb_reg.fit(X_train_fold, y_train_fold)
    xgb_train_predictions += xgb_reg.predict(X_train_full) / kf.n_splits  # Averaged predictions for whole training set
    xgb_test_predictions += xgb_reg.predict(X_test) / kf.n_splits  # Averaged predictions for test set
    
    # Train CatBoost and make predictions on whole training data and X_test
    catboost_reg.fit(X_train_fold, y_train_fold)
    catboost_train_predictions += catboost_reg.predict(X_train_full) / kf.n_splits  # Averaged predictions for whole training set
    catboost_test_predictions += catboost_reg.predict(X_test) / kf.n_splits  # Averaged predictions for test set
    
    # Train LightGBM XT and make predictions on whole training data and X_test
    lgbmxt_reg.fit(X_train_fold, y_train_fold)
    lgbmxt_train_predictions += lgbmxt_reg.predict(X_train_full) / kf.n_splits  # Averaged predictions for whole training set
    lgbmxt_test_predictions += lgbmxt_reg.predict(X_test) / kf.n_splits  # Averaged predictions for test set
    
    # Train LightGBM and make predictions on whole training data and X_test
    lightgbm_reg.fit(X_train_fold, y_train_fold)
    lightgbm_train_predictions += lightgbm_reg.predict(X_train_full) / kf.n_splits  # Averaged predictions for whole training set
    lightgbm_test_predictions += lightgbm_reg.predict(X_test) / kf.n_splits  # Averaged predictions for test set


# Concatenate original features and model predictions correctly (horizontally)
stacked_features = np.concatenate([X_train_full, rf_train_predictions.reshape(-1, 1), xgb_train_predictions.reshape(-1, 1), 
                                   catboost_train_predictions.reshape(-1, 1), lgbmxt_train_predictions.reshape(-1, 1),
                                   lightgbm_train_predictions.reshape(-1, 1)], axis=1)

# Convert stacked features and targets to PyTorch tensors
X_train_tensor = torch.tensor(stacked_features, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_full, dtype=torch.float32).view(-1, 1)

# Define the Custom ANN Model
class RegressionModel(nn.Module):
    def __init__(self, input_dim, layers, neurons):
        super(RegressionModel, self).__init__()
        self.layers = nn.ModuleList()
        self.layers.append(nn.Linear(input_dim, neurons))
        self.layers.append(nn.ReLU())
        for _ in range(layers - 1):
            self.layers.append(nn.Linear(neurons, neurons))
            self.layers.append(nn.ReLU())
        self.layers.append(nn.Linear(neurons, 1))

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

# Custom loss function using the fitted parameters
def custom_loss(outputs, targets, inputs, a, b, e, d):
    mse_loss = nn.MSELoss()(outputs, targets)
    
    AGE = inputs[:, 0]  # Assuming AGE is the first feature
    wb = inputs[:, -12]  # Assuming wb is the seventh feature from the end
    
    AGE = torch.clamp(AGE, min=1e-6)  # Clamp to avoid log of zero
    
    fc_pred = (a * torch.log(AGE) + b) * (e * torch.pow(AGE, d)) ** (-wb)
    residual = torch.abs(outputs - fc_pred.unsqueeze(1))
    residual = torch.nan_to_num(residual, nan=0.0, posinf=1e10, neginf=-1e10)
    
    mean_square_residual = torch.mean(residual ** 2)
    if mean_square_residual.item() > 0:
        residual_normalized = residual * torch.sqrt(mse_loss / mean_square_residual)
    else:
        residual_normalized = residual

    total_loss = 0.5 * mse_loss + 0.5 * torch.mean(residual_normalized)
    return total_loss

# Training function for the meta model
def train_model(model, optimizer, Xtrain, ytrain, epochs=300, batch_size=24, a=None, b=None, e=None, d=None):
    dataset = torch.utils.data.TensorDataset(Xtrain, ytrain)
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)
    
    for epoch in range(epochs):
        model.train()
        for inputs, targets in dataloader:
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = custom_loss(outputs, targets, inputs, a, b, e, d)
            loss.backward()
            optimizer.step()


# Hyperparameters for the custom model
params = {'batch_size': 24, 'layers': 3, 'neurons': 232, 'learn_rate': 0.0076}

# Build the custom ANN model
model = RegressionModel(input_dim=stacked_features.shape[1], layers=params['layers'], neurons=params['neurons'])
optimizer = optim.Adam(model.parameters(), lr=params['learn_rate'])

# Train the custom meta model
train_model(model, optimizer, X_train_tensor, y_train_tensor, epochs=300, batch_size=params['batch_size'], a=40.50, b=15.29, e=6.49, d=0.36)



# Concatenate original test set features and test set predictions for final evaluation

stacked_test_features = np.concatenate([X_test, rf_test_predictions.reshape(-1, 1), xgb_test_predictions.reshape(-1, 1), 
                                   catboost_test_predictions.reshape(-1, 1), lgbmxt_test_predictions.reshape(-1, 1),
                                   lightgbm_test_predictions.reshape(-1, 1)], axis=1)

# Convert test features to PyTorch tensor
X_test_tensor = torch.tensor(stacked_test_features, dtype=torch.float32)

# Use the custom meta-model to predict on the test set
model.eval()
with torch.no_grad():
    y_pred_stacked = model(X_test_tensor).numpy().flatten()

# Evaluate the performance
mse = mean_squared_error(y_test, y_pred_stacked)
r2 = r2_score(y_test, y_pred_stacked)

# Display the results
print(f"Stacking Regressor with Custom ANN Meta-Model Report:\nMSE: {mse:.4f}\nR²: {r2:.4f}")


NameError: name 'nan' is not defined

In [45]:
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold, train_test_split
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error, r2_score
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

# Load dataset
df = pd.read_excel(r'updated_fc_predictions.xlsx', sheet_name='Sheet1')
df.dropna(inplace=True)  # Drop rows with any NaN values

X = df[['AGE', 'PC', 'PC_TYPE', 'FA', 'SS', 'SF', 'FAGG', 'CAGG', 'WATER', 'AEA', 'WR_HR', 'WR', 'ACC', 
        'VOID', 'w/b', 'b/a', 'CAGG%', 'FAGG%', 'FA%', 'SS%', 'SF%']].values

y = df['fc (MPa)'].values

# Split the data into training and testing sets (20% test data)
X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Bagging - Resample 80% of the training data
X_train, _, y_train, _ = train_test_split(X_train_full, y_train_full, train_size=0.8, random_state=42)

from sklearn.model_selection import KFold
import numpy as np

# Define cross-validation strategy
kf = KFold(n_splits=10, shuffle=True, random_state=42)

# Initialize arrays to store averaged predictions on the training and test data
rf_train_predictions = np.zeros(len(X_train_full))
xgb_train_predictions = np.zeros(len(X_train_full))
catboost_train_predictions = np.zeros(len(X_train_full))
lgbmxt_train_predictions = np.zeros(len(X_train_full))
lightgbm_train_predictions = np.zeros(len(X_train_full))
ann_train_predictions = np.zeros(len(X_train_full))

rf_test_predictions = np.zeros(len(X_test))
xgb_test_predictions = np.zeros(len(X_test))
catboost_test_predictions = np.zeros(len(X_test))
lgbmxt_test_predictions = np.zeros(len(X_test))
lightgbm_test_predictions = np.zeros(len(X_test))

ann_test_predictions = np.zeros(len(X_test))
# Initialize base models
# rf_reg = RandomForestRegressor(n_estimators=200, random_state=42)
# xgb_reg = XGBRegressor(random_state=42)
# catboost_reg = CatBoostRegressor(silent=True, random_state=42)
# lgbmxt_reg = LGBMRegressor(boosting_type='goss', random_state=42)
# lightgbm_reg = LGBMRegressor(random_state=42)


rf_reg = RandomForestRegressor(bootstrap=False, max_depth=30, max_features='sqrt',
                      min_samples_split=10, n_estimators=1800)
xgb_reg = XGBRegressor(base_score=0.5, booster='gbtree', callbacks=None,
             colsample_bylevel=1, colsample_bynode=1, colsample_bytree=0.7,
             early_stopping_rounds=None, enable_categorical=False,
             eval_metric=None, gamma=0.0, gpu_id=-1, grow_policy='depthwise',
             importance_type=None, interaction_constraints='',
             learning_rate=0.09, max_bin=256, max_cat_to_onehot=4,
             max_delta_step=0, max_depth=5, max_leaves=0, min_child_weight=1, monotone_constraints='()', n_estimators=500, n_jobs=0,
             num_parallel_tree=1, predictor='auto', random_state=42, reg_alpha=0,
             reg_lambda=1)
catboost_reg = CatBoostRegressor(silent=True,learning_rate=0.08, l2_leaf_reg= 6, iterations=500, depth=6, border_count=48, random_state=42)
lgbmxt_reg = LGBMRegressor(boosting_type='goss', reg_lambda=0.3, reg_alpha=0.4, num_leaves=40, n_estimators=500, min_child_samples= 45, 
learning_rate= 0.06, colsample_bytree=0.5,random_state=42)
ann_reg = MLPRegressor(solver= 'adam', max_iter=300, learning_rate_init=0.001, learning_rate='adaptive', 
hidden_layer_sizes=(50,), alpha=0.01, activation='relu', random_state=42)

# Manual cross-validation to get predictions for both train and test sets
for train_idx, val_idx in kf.split(X_train_full):
    # Split into training and validation based on the fold
    X_train_fold, X_val_fold = X_train_full[train_idx], X_train_full[val_idx]
    y_train_fold, y_val_fold = y_train_full[train_idx], y_train_full[val_idx]
    
    # Train Random Forest and make predictions on whole training data and X_test
    rf_reg.fit(X_train_fold, y_train_fold)
    rf_train_predictions += rf_reg.predict(X_train_full) / kf.n_splits  # Averaged predictions for whole training set
    rf_test_predictions += rf_reg.predict(X_test) / kf.n_splits  # Averaged predictions for test set
    
    # Train XGBoost and make predictions on whole training data and X_test
    xgb_reg.fit(X_train_fold, y_train_fold)
    xgb_train_predictions += xgb_reg.predict(X_train_full) / kf.n_splits  # Averaged predictions for whole training set
    xgb_test_predictions += xgb_reg.predict(X_test) / kf.n_splits  # Averaged predictions for test set
    
    # Train CatBoost and make predictions on whole training data and X_test
    catboost_reg.fit(X_train_fold, y_train_fold)
    catboost_train_predictions += catboost_reg.predict(X_train_full) / kf.n_splits  # Averaged predictions for whole training set
    catboost_test_predictions += catboost_reg.predict(X_test) / kf.n_splits  # Averaged predictions for test set
    
    # Train LightGBM XT and make predictions on whole training data and X_test
    lgbmxt_reg.fit(X_train_fold, y_train_fold)
    lgbmxt_train_predictions += lgbmxt_reg.predict(X_train_full) / kf.n_splits  # Averaged predictions for whole training set
    lgbmxt_test_predictions += lgbmxt_reg.predict(X_test) / kf.n_splits  # Averaged predictions for test set
    
    # Train LightGBM and make predictions on whole training data and X_test
    lightgbm_reg.fit(X_train_fold, y_train_fold)
    lightgbm_train_predictions += lightgbm_reg.predict(X_train_full) / kf.n_splits  # Averaged predictions for whole training set
    lightgbm_test_predictions += lightgbm_reg.predict(X_test) / kf.n_splits  # Averaged predictions for test set

    ann_reg.fit(X_train_fold, y_train_fold)
    ann_train_predictions += ann_reg.predict(X_train_full) / kf.n_splits  # Averaged predictions for whole training set
    ann_test_predictions += ann_reg.predict(X_test) / kf.n_splits  # Averaged predictions for test set


# Concatenate original features and model predictions correctly (horizontally)
stacked_features = np.concatenate([X_train_full, rf_train_predictions.reshape(-1, 1), xgb_train_predictions.reshape(-1, 1), 
                                   catboost_train_predictions.reshape(-1, 1), lgbmxt_train_predictions.reshape(-1, 1),
                                   lightgbm_train_predictions.reshape(-1, 1),ann_train_predictions.reshape(-1, 1)], axis=1)

# Convert to PyTorch tensors for the meta-model
X_train_tensor = torch.tensor(stacked_features, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_full, dtype=torch.float32).view(-1, 1)

# Define the Custom ANN Meta-Model
class RegressionModel(nn.Module):
    def __init__(self, input_dim, layers, neurons):
        super(RegressionModel, self).__init__()
        self.layers = nn.ModuleList()
        self.layers.append(nn.Linear(input_dim, neurons))
        self.layers.append(nn.ReLU())
        for _ in range(layers - 1):
            self.layers.append(nn.Linear(neurons, neurons))
            self.layers.append(nn.ReLU())
        self.layers.append(nn.Linear(neurons, 1))

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

# Custom loss function using the fitted parameters
def custom_loss(outputs, targets, inputs, a, b, e, d):
    mse_loss = nn.MSELoss()(outputs, targets)
    
    AGE = inputs[:, 0]  # Assuming AGE is the first feature
    wb = inputs[:, -13]  # Assuming wb is the 12th from the end (change if necessary)
    
    AGE = torch.clamp(AGE, min=1e-6)  # Clamp to avoid log of zero
    
    fc_pred = (a * torch.log(AGE) + b) * (e * torch.pow(AGE, d)) ** (-wb)
    residual = torch.abs(outputs - fc_pred.unsqueeze(1))
    residual = torch.nan_to_num(residual, nan=0.0, posinf=1e10, neginf=-1e10)
    
    mean_square_residual = torch.mean(residual ** 2)
    if mean_square_residual.item() > 0:
        residual_normalized = residual * torch.sqrt(mse_loss / mean_square_residual)
    else:
        residual_normalized = residual

    total_loss = 0.5 * mse_loss + 0.5 * torch.mean(residual_normalized)
    return total_loss

# Training function for the meta-model
def train_model(model, optimizer, Xtrain, ytrain, epochs=300, batch_size=24, a=None, b=None, e=None, d=None):
    dataset = torch.utils.data.TensorDataset(Xtrain, ytrain)
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)
    
    for epoch in range(epochs):
        model.train()
        for inputs, targets in dataloader:
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = custom_loss(outputs, targets, inputs, a, b, e, d)
            loss.backward()
            optimizer.step()

# Hyperparameters for the custom model
params = {'batch_size': 24, 'layers': 3, 'neurons': 232, 'learn_rate': 0.0076}

# Set up K-Fold cross-validation (e.g., 5 folds)
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Lists to store cross-validation results
mse_scores = []
r2_scores = []
mse_scores_val=[]
r2_scores_val=[]

# Concatenate original test set features and test set predictions for final evaluation
stacked_test_features = np.concatenate([X_test, rf_test_predictions.reshape(-1, 1), xgb_test_predictions.reshape(-1, 1), 
                                   catboost_test_predictions.reshape(-1, 1), lgbmxt_test_predictions.reshape(-1, 1),
                                   lightgbm_test_predictions.reshape(-1, 1),ann_test_predictions.reshape(-1, 1)], axis=1)

# Convert test features to PyTorch tensor
X_test_tensor = torch.tensor(stacked_test_features, dtype=torch.float32)


# Cross-validation for the meta-model using stacked features
for fold, (train_idx, val_idx) in enumerate(kf.split(X_train_tensor)):
    print(f"Fold {fold + 1}")

    # Split the stacked features into training and validation sets
    X_train_fold = X_train_tensor[train_idx]
    y_train_fold = y_train_tensor[train_idx]
    X_val_fold = X_train_tensor[val_idx]
    y_val_fold = y_train_tensor[val_idx]

    # Build a fresh custom ANN model for each fold
    model = RegressionModel(input_dim=stacked_features.shape[1], layers=params['layers'], neurons=params['neurons'])
    optimizer = optim.Adam(model.parameters(), lr=params['learn_rate'])

    # Train the custom ANN meta-model on the training fold
    train_model(model, optimizer, X_train_fold, y_train_fold, epochs=300, batch_size=params['batch_size'], 
                a=40.50, b=15.29, e=6.49, d=0.36)

    # Evaluate the meta-model on the validation fold
    model.eval()
    with torch.no_grad():
        y_pred_test = model(X_test_tensor).numpy().flatten()
        y_pred_val = model(X_val_fold).numpy().flatten()

    mse_val = mean_squared_error(y_val_fold, y_pred_val)
    r2_val = r2_score(y_val_fold, y_pred_val)

    # mse_scores.append(mse_val)
    # r2_scores.append(r2_val)

    mse_scores_val.append(mse_val)
    r2_scores_val.append(r2_val)

    # Calculate performance metrics for this fold
    mse_test = mean_squared_error(y_test, y_pred_test)
    r2_test = r2_score(y_test, y_pred_test)

    # Append the metrics to the list
    mse_scores.append(mse_test)
    r2_scores.append(r2_test)


# Calculate average MSE and R² over all folds
avg_mse = np.mean(mse_scores)
avg_r2 = np.mean(r2_scores)

# Display the cross-validation results
print(f"Cross-Validation Results for Custom ANN Meta-Model:\nAverage MSE: {avg_mse:.4f}\nAverage R²: {avg_r2:.4f}")


/Users/zhangtianjie/opt/anaconda3/envs/ag/lib/python3.10/site-packages/xgboost/core.py:158: UserWarning: [11:38:24] WARNING: /Users/runner/miniforge3/conda-bld/xgboost-split_1727634900191/work/src/learner.cc:740: 
Parameters: { "predictor" } are not used.

  warnings.warn(smsg, UserWarning)


[LightGBM] [Warning] Found boosting=goss. For backwards compatibility reasons, LightGBM interprets this as boosting=gbdt, data_sample_strategy=goss.To suppress this warning, set data_sample_strategy=goss instead.
[LightGBM] [Warning] Found boosting=goss. For backwards compatibility reasons, LightGBM interprets this as boosting=gbdt, data_sample_strategy=goss.To suppress this warning, set data_sample_strategy=goss instead.
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000559 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2870
[LightGBM] [Info] Number of data points in the train set: 4861, number of used features: 21
[LightGBM] [Info] Using GOSS
[LightGBM] [Info] Start training from score 41.564665
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive 

/Users/zhangtianjie/opt/anaconda3/envs/ag/lib/python3.10/site-packages/xgboost/core.py:158: UserWarning: [11:38:45] WARNING: /Users/runner/miniforge3/conda-bld/xgboost-split_1727634900191/work/src/learner.cc:740: 
Parameters: { "predictor" } are not used.

  warnings.warn(smsg, UserWarning)


[LightGBM] [Warning] Found boosting=goss. For backwards compatibility reasons, LightGBM interprets this as boosting=gbdt, data_sample_strategy=goss.To suppress this warning, set data_sample_strategy=goss instead.
[LightGBM] [Warning] Found boosting=goss. For backwards compatibility reasons, LightGBM interprets this as boosting=gbdt, data_sample_strategy=goss.To suppress this warning, set data_sample_strategy=goss instead.
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000727 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2889
[LightGBM] [Info] Number of data points in the train set: 4861, number of used features: 21
[LightGBM] [Info] Using GOSS
[LightGBM] [Info] Start training from score 41.591387
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive 

/Users/zhangtianjie/opt/anaconda3/envs/ag/lib/python3.10/site-packages/xgboost/core.py:158: UserWarning: [11:39:06] WARNING: /Users/runner/miniforge3/conda-bld/xgboost-split_1727634900191/work/src/learner.cc:740: 
Parameters: { "predictor" } are not used.

  warnings.warn(smsg, UserWarning)


...

[LightGBM] [Warning] Found boosting=goss. For backwards compatibility reasons, LightGBM interprets this as boosting=gbdt, data_sample_strategy=goss.To suppress this warning, set data_sample_strategy=goss instead.
[LightGBM] [Warning] Found boosting=goss. For backwards compatibility reasons, LightGBM interprets this as boosting=gbdt, data_sample_strategy=goss.To suppress this warning, set data_sample_strategy=goss instead.
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000548 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2890
[LightGBM] [Info] Number of data points in the train set: 4862, number of used features: 21
[LightGBM] [Info] Using GOSS
[LightGBM] [Info] Start training from score 41.652398
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive 

/Users/zhangtianjie/opt/anaconda3/envs/ag/lib/python3.10/site-packages/xgboost/core.py:158: UserWarning: [11:41:12] WARNING: /Users/runner/miniforge3/conda-bld/xgboost-split_1727634900191/work/src/learner.cc:740: 
Parameters: { "predictor" } are not used.

  warnings.warn(smsg, UserWarning)


[LightGBM] [Warning] Found boosting=goss. For backwards compatibility reasons, LightGBM interprets this as boosting=gbdt, data_sample_strategy=goss.To suppress this warning, set data_sample_strategy=goss instead.
[LightGBM] [Warning] Found boosting=goss. For backwards compatibility reasons, LightGBM interprets this as boosting=gbdt, data_sample_strategy=goss.To suppress this warning, set data_sample_strategy=goss instead.
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000591 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2881
[LightGBM] [Info] Number of data points in the train set: 4862, number of used features: 21
[LightGBM] [Info] Using GOSS
[LightGBM] [Info] Start training from score 41.786458
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive 

/Users/zhangtianjie/opt/anaconda3/envs/ag/lib/python3.10/site-packages/xgboost/core.py:158: UserWarning: [11:41:31] WARNING: /Users/runner/miniforge3/conda-bld/xgboost-split_1727634900191/work/src/learner.cc:740: 
Parameters: { "predictor" } are not used.

  warnings.warn(smsg, UserWarning)


[LightGBM] [Warning] Found boosting=goss. For backwards compatibility reasons, LightGBM interprets this as boosting=gbdt, data_sample_strategy=goss.To suppress this warning, set data_sample_strategy=goss instead.
[LightGBM] [Warning] Found boosting=goss. For backwards compatibility reasons, LightGBM interprets this as boosting=gbdt, data_sample_strategy=goss.To suppress this warning, set data_sample_strategy=goss instead.
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000592 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2880
[LightGBM] [Info] Number of data points in the train set: 4862, number of used features: 21
[LightGBM] [Info] Using GOSS
[LightGBM] [Info] Start training from score 41.650829
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive 

In [46]:
print(mse_scores_val)
print(r2_scores_val)

[10.866176, 15.034351, 13.227545, 18.043472, 12.85151]
[0.9474806189537048, 0.9207650423049927, 0.9300943613052368, 0.9093542695045471, 0.9284238219261169]


In [47]:
print(mse_scores)
print(r2_scores)

[45.70802839738857, 47.90014985135813, 46.263481139484675, 46.60803706969507, 45.93393341187414]
[0.7672959991354238, 0.7561356964357577, 0.7644681354119395, 0.7627139677899084, 0.7661458948205072]


In [ ]:
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold, train_test_split
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error, r2_score
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

# Load dataset
df = pd.read_excel(r'updated_fc_predictions.xlsx', sheet_name='Sheet1')
df.dropna(inplace=True)  # Drop rows with any NaN values

X = df[['AGE', 'PC', 'PC_TYPE', 'FA', 'SS', 'SF', 'FAGG', 'CAGG', 'WATER', 'AEA', 'WR_HR', 'WR', 'ACC', 
        'VOID', 'w/b', 'b/a', 'CAGG%', 'FAGG%', 'FA%', 'SS%', 'SF%']].values

y = df['fc (MPa)'].values

# Split the data into training and testing sets (20% test data)
X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Bagging - Resample 80% of the training data
X_train, _, y_train, _ = train_test_split(X_train_full, y_train_full, train_size=0.8, random_state=42)

from sklearn.model_selection import KFold
import numpy as np

# Define cross-validation strategy
kf = KFold(n_splits=10, shuffle=True, random_state=42)

# Initialize arrays to store averaged predictions on the training and test data
rf_train_predictions = np.zeros(len(X_train_full))
xgb_train_predictions = np.zeros(len(X_train_full))
catboost_train_predictions = np.zeros(len(X_train_full))
lgbmxt_train_predictions = np.zeros(len(X_train_full))
lightgbm_train_predictions = np.zeros(len(X_train_full))
ann_train_predictions = np.zeros(len(X_train_full))

rf_test_predictions = np.zeros(len(X_test))
xgb_test_predictions = np.zeros(len(X_test))
catboost_test_predictions = np.zeros(len(X_test))
lgbmxt_test_predictions = np.zeros(len(X_test))
lightgbm_test_predictions = np.zeros(len(X_test))

ann_test_predictions = np.zeros(len(X_test))
# Initialize base models
# rf_reg = RandomForestRegressor(n_estimators=200, random_state=42)
# xgb_reg = XGBRegressor(random_state=42)
# catboost_reg = CatBoostRegressor(silent=True, random_state=42)
# lgbmxt_reg = LGBMRegressor(boosting_type='goss', random_state=42)
# lightgbm_reg = LGBMRegressor(random_state=42)


rf_reg = RandomForestRegressor(bootstrap=False, max_depth=30, max_features='sqrt',
                      min_samples_split=10, n_estimators=1800)
xgb_reg = XGBRegressor(base_score=0.5, booster='gbtree', callbacks=None,
             colsample_bylevel=1, colsample_bynode=1, colsample_bytree=0.7,
             early_stopping_rounds=None, enable_categorical=False,
             eval_metric=None, gamma=0.0, gpu_id=-1, grow_policy='depthwise',
             importance_type=None, interaction_constraints='',
             learning_rate=0.09, max_bin=256, max_cat_to_onehot=4,
             max_delta_step=0, max_depth=5, max_leaves=0, min_child_weight=1, monotone_constraints='()', n_estimators=500, n_jobs=0,
             num_parallel_tree=1, predictor='auto', random_state=42, reg_alpha=0,
             reg_lambda=1)
catboost_reg = CatBoostRegressor(silent=True,learning_rate=0.08, l2_leaf_reg= 6, iterations=500, depth=6, border_count=48, random_state=42)
lgbmxt_reg = LGBMRegressor(boosting_type='goss', reg_lambda=0.3, reg_alpha=0.4, num_leaves=40, n_estimators=500, min_child_samples= 45, 
learning_rate= 0.06, colsample_bytree=0.5,random_state=42)
ann_reg = MLPRegressor(solver= 'adam', max_iter=300, learning_rate_init=0.001, learning_rate='adaptive', 
hidden_layer_sizes=(50,), alpha=0.01, activation='relu', random_state=42)

# Manual cross-validation to get predictions for both train and test sets
for train_idx, val_idx in kf.split(X_train_full):
    # Split into training and validation based on the fold
    X_train_fold, X_val_fold = X_train_full[train_idx], X_train_full[val_idx]
    y_train_fold, y_val_fold = y_train_full[train_idx], y_train_full[val_idx]
    
    # Train Random Forest and make predictions on whole training data and X_test
    rf_reg.fit(X_train_fold, y_train_fold)
    rf_train_predictions += rf_reg.predict(X_train_full) / kf.n_splits  # Averaged predictions for whole training set
    rf_test_predictions += rf_reg.predict(X_test) / kf.n_splits  # Averaged predictions for test set
    
    # Train XGBoost and make predictions on whole training data and X_test
    xgb_reg.fit(X_train_fold, y_train_fold)
    xgb_train_predictions += xgb_reg.predict(X_train_full) / kf.n_splits  # Averaged predictions for whole training set
    xgb_test_predictions += xgb_reg.predict(X_test) / kf.n_splits  # Averaged predictions for test set
    
    # Train CatBoost and make predictions on whole training data and X_test
    catboost_reg.fit(X_train_fold, y_train_fold)
    catboost_train_predictions += catboost_reg.predict(X_train_full) / kf.n_splits  # Averaged predictions for whole training set
    catboost_test_predictions += catboost_reg.predict(X_test) / kf.n_splits  # Averaged predictions for test set
    
    # Train LightGBM XT and make predictions on whole training data and X_test
    lgbmxt_reg.fit(X_train_fold, y_train_fold)
    lgbmxt_train_predictions += lgbmxt_reg.predict(X_train_full) / kf.n_splits  # Averaged predictions for whole training set
    lgbmxt_test_predictions += lgbmxt_reg.predict(X_test) / kf.n_splits  # Averaged predictions for test set
    
    # Train LightGBM and make predictions on whole training data and X_test
    lightgbm_reg.fit(X_train_fold, y_train_fold)
    lightgbm_train_predictions += lightgbm_reg.predict(X_train_full) / kf.n_splits  # Averaged predictions for whole training set
    lightgbm_test_predictions += lightgbm_reg.predict(X_test) / kf.n_splits  # Averaged predictions for test set

    ann_reg.fit(X_train_fold, y_train_fold)
    ann_train_predictions += ann_reg.predict(X_train_full) / kf.n_splits  # Averaged predictions for whole training set
    ann_test_predictions += ann_reg.predict(X_test) / kf.n_splits  # Averaged predictions for test set


# Concatenate original features and model predictions correctly (horizontally)
stacked_features = np.concatenate([X_train_full, rf_train_predictions.reshape(-1, 1), xgb_train_predictions.reshape(-1, 1), 
                                   catboost_train_predictions.reshape(-1, 1), lgbmxt_train_predictions.reshape(-1, 1),
                                   lightgbm_train_predictions.reshape(-1, 1),ann_train_predictions.reshape(-1, 1)], axis=1)

# Convert to PyTorch tensors for the meta-model
X_train_tensor = torch.tensor(stacked_features, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_full, dtype=torch.float32).view(-1, 1)

class RegressionModel(nn.Module):
    def __init__(self, input_dim, layers, neurons):
        super(RegressionModel, self).__init__()
        self.layers = nn.ModuleList()
        self.layers.append(nn.Linear(input_dim, neurons))
        self.layers.append(nn.ReLU())
        for _ in range(layers - 1):
            self.layers.append(nn.Linear(neurons, neurons))
            self.layers.append(nn.ReLU())
        self.layers.append(nn.Linear(neurons, 1))
    
    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return torch.nan_to_num(x, nan=-1e6, posinf=1e6, neginf=-1e6)

# Step 6: Custom loss function using the fitted parameters
def custom_loss(outputs, targets, inputs):
    if torch.isnan(outputs).any():
        print("NaN found in outputs")

    mse_loss = nn.MSELoss()(outputs, targets)
    
    # Extract features needed for the fitted equation
    t = inputs[:, 0]  # AGE is the first feature
    w_c = inputs[:, 15]  # w/b is the 16th feature
    
    # Clamp t and w_c to prevent numerical issues
    t = torch.clamp(t, min=1e-6, max=1e3)
    w_c = torch.clamp(w_c, min=1e-6, max=1e3)
    

    PC = inputs[:, 1]
    PC_TYPE = inputs[:, 2]
    FA = inputs[:, 3]
    SS = inputs[:, 4]
    SF = inputs[:, 5]
    FAGG = inputs[:, 6]
    CAGG = inputs[:, 7]
    WATER = inputs[:, 8]
    AEA = inputs[:, 9]
    WR_HR = inputs[:, 10]
    WR = inputs[:, 11]
    ACC = inputs[:, 12]
    VOID = inputs[:, 13]
    ba_ratio = inputs[:, 15]
    CAGG_pct = inputs[:, 16]
    FAGG_pct = inputs[:, 17]
    FA_pct = inputs[:, 18]
    SS_pct = inputs[:, 19]
    SF_pct = inputs[:, 20]
    
    # Fitted parameters (as per your values)
    a = [ 0,2.0265,  4.63e-01,  1.85, -8.78e-01, -1.136e-01,  5.799e-01,  1.761e-02,  1.367e-01,
     -5.81e-01,  2.221,  2.768e-01,  2.35e-01, -7.69e-03, -1.323,  8.92e+02,  4.794e+02,
     -8.845e+02, -6.281e+02,  1.009e+03,  5.145e+02, 3.62e+02]
    
    b = [ 0,3.71e-01, -5.4251e-02,  1.5051e-01, -1.0079e+00, -3.778e-01,  5.802e-01,  1.085e-01,  1.658e-01,
     -6.12e-01,  4.771,  1.563e-01,  6.11e-02, -1.557e-02, -5.202e-01,  4.233e+02,  1.324e+03,
     -7.396e+02, -5.880e+02,  6.944e+02,  2.6335e+02, -2.829e+02]

    # Compute the fitted equation for A and B
    A = a[0] + a[1]*t + a[2]*PC + a[3]*PC_TYPE + a[4]*FA + a[5]*SS + a[6]*SF + a[7]*FAGG + a[8]*CAGG + a[9]*WATER + a[10]*AEA + a[11]*WR_HR + a[12]*WR + a[13]*ACC + a[14]*VOID + a[15]*w_c + a[16]*ba_ratio + a[17]*CAGG_pct + a[18]*FAGG_pct + a[19]*FA_pct + a[20]*SS_pct + a[21]*SF_pct
    B = b[0] + b[1]*t + b[2]*PC + b[3]*PC_TYPE + b[4]*FA + b[5]*SS + b[6]*SF + b[7]*FAGG + b[8]*CAGG + b[9]*WATER + b[10]*AEA + b[11]*WR_HR + b[12]*WR + b[13]*ACC + b[14]*VOID + b[15]*w_c + b[16]*ba_ratio + b[17]*CAGG_pct + b[18]*FAGG_pct + b[19]*FA_pct + b[20]*SS_pct + b[21]*SF_pct


    A = torch.clamp(A, min=-1e6, max=1e6)
    B = torch.clamp(B, min=1e-6, max=1e6)  # Prevent very small or negative values
    fitted_fc = torch.clamp(A * B ** (-w_c), min=-1e6, max=1e3)

    # Debugging
    if torch.isnan(A).any():
        print("NaN found in A:", A)
    if torch.isnan(B).any():
        print("NaN found in B:", B)
    if torch.isnan(fitted_fc).any():
        print("NaN found in fitted_fc:", fitted_fc)
    if torch.isnan(outputs).any():
        print("NaN found in outputs:", outputs)
    # Calculate fitted fc
    #fitted_fc = A * B ** (-w_c)
    #fitted_fc = torch.clamp(A * B ** (-w_c), min=-1e6, max=1e3)

    #print(fitted_fc)
    #if torch.isnan(fitted_fc).any():
        #print("NaN found in fc")
    
    # Calculate the residual between predicted and fitted equation
    #residual = torch.abs(outputs - fitted_fc.unsqueeze(1))
    #residual = torch.clamp(residual, min=1e-6, max=1e3)  # Clip to avoid extreme values
    residual = torch.clamp(torch.abs(outputs - fitted_fc.unsqueeze(1)), min=1e-6, max=1e3)
    
    
    # Normalize residual
    
    # Normalize residual by comparing its mean square with the MSE
    mean_square_residual = torch.mean(residual ** 2)
    if torch.isnan(mse_loss).any():
        print("NaN found in MSE loss")
    mean_square_residual = torch.clamp(mean_square_residual, min=1e-6,max=1e3)  # Prevent division by 0
    residual_normalized = residual * torch.sqrt(mse_loss / mean_square_residual)
    
    # Combine MSE loss and the normalized residual
    total_loss = 0.5 * mse_loss + 0.5 * torch.mean(residual_normalized)
    if torch.isnan(total_loss).any():
        print("NaN found in total loss")
    
    return total_loss

# Training function
def train_model(model, optimizer, Xtrain, ytrain, epochs=300, batch_size=24):
    dataset = torch.utils.data.TensorDataset(Xtrain, ytrain)
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)
    
    for epoch in range(epochs):
        model.train()
        for inputs, targets in dataloader:
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = custom_loss(outputs, targets, inputs)
            loss.backward()
            optimizer.step()

# Hyperparameters for the custom model
params = {'batch_size': 24, 'layers': 3, 'neurons': 232, 'learn_rate': 0.0076}

# Set up K-Fold cross-validation (e.g., 5 folds)
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Lists to store cross-validation results
mse_scores = []
r2_scores = []
mse_scores_val=[]
r2_scores_val=[]

# Concatenate original test set features and test set predictions for final evaluation
stacked_test_features = np.concatenate([X_test, rf_test_predictions.reshape(-1, 1), xgb_test_predictions.reshape(-1, 1), 
                                   catboost_test_predictions.reshape(-1, 1), lgbmxt_test_predictions.reshape(-1, 1),
                                   lightgbm_test_predictions.reshape(-1, 1),ann_test_predictions.reshape(-1, 1)], axis=1)

# Convert test features to PyTorch tensor
X_test_tensor = torch.tensor(stacked_test_features, dtype=torch.float32)


# Cross-validation for the meta-model using stacked features
for fold, (train_idx, val_idx) in enumerate(kf.split(X_train_tensor)):
    print(f"Fold {fold + 1}")

    # Split the stacked features into training and validation sets
    X_train_fold = X_train_tensor[train_idx]
    y_train_fold = y_train_tensor[train_idx]
    X_val_fold = X_train_tensor[val_idx]
    y_val_fold = y_train_tensor[val_idx]

    # Build a fresh custom ANN model for each fold
    model = RegressionModel(input_dim=stacked_features.shape[1], layers=params['layers'], neurons=params['neurons'])
    optimizer = optim.Adam(model.parameters(), lr=params['learn_rate'])

    # Train the custom ANN meta-model on the training fold
    train_model(model, optimizer, X_train_fold, y_train_fold, epochs=300, batch_size=params['batch_size'])

    # Evaluate the meta-model on the validation fold
    model.eval()
    with torch.no_grad():
        y_pred_test = model(X_test_tensor).numpy().flatten()
        y_pred_val = model(X_val_fold).numpy().flatten()

    mse_val = mean_squared_error(y_val_fold, y_pred_val)
    r2_val = r2_score(y_val_fold, y_pred_val)

    # mse_scores.append(mse_val)
    # r2_scores.append(r2_val)

    mse_scores_val.append(mse_val)
    r2_scores_val.append(r2_val)

    # Calculate performance metrics for this fold
    mse_test = mean_squared_error(y_test, y_pred_test)
    r2_test = r2_score(y_test, y_pred_test)

    # Append the metrics to the list
    mse_scores.append(mse_test)
    r2_scores.append(r2_test)


# Calculate average MSE and R² over all folds
avg_mse = np.mean(mse_scores)
avg_r2 = np.mean(r2_scores)

# Display the cross-validation results
print(f"Cross-Validation Results for Custom ANN Meta-Model:\nAverage MSE: {avg_mse:.4f}\nAverage R²: {avg_r2:.4f}")


/Users/zhangtianjie/opt/anaconda3/envs/ag/lib/python3.10/site-packages/xgboost/core.py:158: UserWarning: [12:52:44] WARNING: /Users/runner/miniforge3/conda-bld/xgboost-split_1727634900191/work/src/learner.cc:740: 
Parameters: { "predictor" } are not used.

  warnings.warn(smsg, UserWarning)


[LightGBM] [Warning] Found boosting=goss. For backwards compatibility reasons, LightGBM interprets this as boosting=gbdt, data_sample_strategy=goss.To suppress this warning, set data_sample_strategy=goss instead.
[LightGBM] [Warning] Found boosting=goss. For backwards compatibility reasons, LightGBM interprets this as boosting=gbdt, data_sample_strategy=goss.To suppress this warning, set data_sample_strategy=goss instead.
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000603 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2870
[LightGBM] [Info] Number of data points in the train set: 4861, number of used features: 21
[LightGBM] [Info] Using GOSS
[LightGBM] [Info] Start training from score 41.564665
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive 

/Users/zhangtianjie/opt/anaconda3/envs/ag/lib/python3.10/site-packages/xgboost/core.py:158: UserWarning: [12:53:05] WARNING: /Users/runner/miniforge3/conda-bld/xgboost-split_1727634900191/work/src/learner.cc:740: 
Parameters: { "predictor" } are not used.

  warnings.warn(smsg, UserWarning)


[LightGBM] [Warning] Found boosting=goss. For backwards compatibility reasons, LightGBM interprets this as boosting=gbdt, data_sample_strategy=goss.To suppress this warning, set data_sample_strategy=goss instead.
[LightGBM] [Warning] Found boosting=goss. For backwards compatibility reasons, LightGBM interprets this as boosting=gbdt, data_sample_strategy=goss.To suppress this warning, set data_sample_strategy=goss instead.
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000898 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2889
[LightGBM] [Info] Number of data points in the train set: 4861, number of used features: 21
[LightGBM] [Info] Using GOSS
[LightGBM] [Info] Start training from score 41.591387
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive 

/Users/zhangtianjie/opt/anaconda3/envs/ag/lib/python3.10/site-packages/xgboost/core.py:158: UserWarning: [12:53:27] WARNING: /Users/runner/miniforge3/conda-bld/xgboost-split_1727634900191/work/src/learner.cc:740: 
Parameters: { "predictor" } are not used.

  warnings.warn(smsg, UserWarning)


...

[LightGBM] [Warning] Found boosting=goss. For backwards compatibility reasons, LightGBM interprets this as boosting=gbdt, data_sample_strategy=goss.To suppress this warning, set data_sample_strategy=goss instead.
[LightGBM] [Warning] Found boosting=goss. For backwards compatibility reasons, LightGBM interprets this as boosting=gbdt, data_sample_strategy=goss.To suppress this warning, set data_sample_strategy=goss instead.
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000544 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2890
[LightGBM] [Info] Number of data points in the train set: 4862, number of used features: 21
[LightGBM] [Info] Using GOSS
[LightGBM] [Info] Start training from score 41.652398
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive 

/Users/zhangtianjie/opt/anaconda3/envs/ag/lib/python3.10/site-packages/xgboost/core.py:158: UserWarning: [12:55:38] WARNING: /Users/runner/miniforge3/conda-bld/xgboost-split_1727634900191/work/src/learner.cc:740: 
Parameters: { "predictor" } are not used.

  warnings.warn(smsg, UserWarning)


[LightGBM] [Warning] Found boosting=goss. For backwards compatibility reasons, LightGBM interprets this as boosting=gbdt, data_sample_strategy=goss.To suppress this warning, set data_sample_strategy=goss instead.
[LightGBM] [Warning] Found boosting=goss. For backwards compatibility reasons, LightGBM interprets this as boosting=gbdt, data_sample_strategy=goss.To suppress this warning, set data_sample_strategy=goss instead.
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000603 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2881
[LightGBM] [Info] Number of data points in the train set: 4862, number of used features: 21
[LightGBM] [Info] Using GOSS
[LightGBM] [Info] Start training from score 41.786458
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive 

/Users/zhangtianjie/opt/anaconda3/envs/ag/lib/python3.10/site-packages/xgboost/core.py:158: UserWarning: [12:55:59] WARNING: /Users/runner/miniforge3/conda-bld/xgboost-split_1727634900191/work/src/learner.cc:740: 
Parameters: { "predictor" } are not used.

  warnings.warn(smsg, UserWarning)


[LightGBM] [Warning] Found boosting=goss. For backwards compatibility reasons, LightGBM interprets this as boosting=gbdt, data_sample_strategy=goss.To suppress this warning, set data_sample_strategy=goss instead.
[LightGBM] [Warning] Found boosting=goss. For backwards compatibility reasons, LightGBM interprets this as boosting=gbdt, data_sample_strategy=goss.To suppress this warning, set data_sample_strategy=goss instead.
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000591 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2880
[LightGBM] [Info] Number of data points in the train set: 4862, number of used features: 21
[LightGBM] [Info] Using GOSS
[LightGBM] [Info] Start training from score 41.650829
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive 